In [17]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict
from pydantic import SecretStr

import os
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
# === Utility Functions ===
def get_secret(key_name: str) -> SecretStr:
    value = os.getenv(key_name)
    if not value:
        raise ValueError(
            f"❌ {key_name} not found. Please set it in your .env file.")
    return SecretStr(value)

In [19]:
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.5,
    api_key=get_secret("GROQ_API_KEY")
)

In [20]:
# create a state
class LLMState(TypedDict):
    question: str
    answer: str

In [21]:
def llm_qa(state: LLMState) -> LLMState:
    # extract the question from the state
    question = state['question']
    
    # create a prompt 
    prompt = f"Answer the following {question}"
    
    # ask that question
    answer = model.invoke(prompt).content 
    
    # update the answer in the state 
    state['answer'] = answer
    return state

In [22]:
# Create our graph
graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [23]:
# execute
initial_state = {'question': 'How far is moon form the earth?'}
final_state = workflow.invoke(initial_state)
print(final_state)

{'question': 'How far is moon form the earth?', 'answer': 'The Moon’s distance from Earth isn’t a single fixed number—it varies because its orbit is slightly elliptical. Here are the key figures:\n\n| Parameter | Approximate Value |\n|-----------|-------------------|\n| **Average (mean) distance** | **384\u202f400\u202fkm** (≈\u202f238\u202f855\u202fmi) |\n| **Perigee (closest approach)** | ~363\u202f300\u202fkm (≈\u202f225\u202f700\u202fmi) |\n| **Apogee (farthest point)** | ~405\u202f500\u202fkm (≈\u202f251\u202f900\u202fmi) |\n| **Orbital period (sidereal month)** | 27.3\u202fdays (≈\u202f655\u202fhours) |\n| **Synodic month (full‑to‑full)** | 29.5\u202fdays (≈\u202f709\u202fhours) |\n\n### Why the distance changes\n- **Elliptical orbit:** The Moon’s orbit has an eccentricity of about 0.0549, so it’s not a perfect circle.\n- **Gravitational perturbations:** The Sun, Earth’s equatorial bulge, and other bodies cause slight variations in the orbit over time.\n\n### Quick reference in e

In [24]:
print(final_state['answer'])

The Moon’s distance from Earth isn’t a single fixed number—it varies because its orbit is slightly elliptical. Here are the key figures:

| Parameter | Approximate Value |
|-----------|-------------------|
| **Average (mean) distance** | **384 400 km** (≈ 238 855 mi) |
| **Perigee (closest approach)** | ~363 300 km (≈ 225 700 mi) |
| **Apogee (farthest point)** | ~405 500 km (≈ 251 900 mi) |
| **Orbital period (sidereal month)** | 27.3 days (≈ 655 hours) |
| **Synodic month (full‑to‑full)** | 29.5 days (≈ 709 hours) |

### Why the distance changes
- **Elliptical orbit:** The Moon’s orbit has an eccentricity of about 0.0549, so it’s not a perfect circle.
- **Gravitational perturbations:** The Sun, Earth’s equatorial bulge, and other bodies cause slight variations in the orbit over time.

### Quick reference in everyday terms
- If you could drive a car non‑stop at 100 km/h, it would take **about 160 days** to cover the average distance.
- Light from the Moon takes **≈ 1.28 seconds** to r